In [19]:
import numpy as np
from numpy import linalg as la
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

Things to do:

- Instatiation of all elements
- Neigbours
- Force calculations


In [20]:
# Number of atoms to take as input.
n = int(input("Enter number of atoms: "))

In [ ]:
dt = 0.5  # The time the simulation progresses in one time step
r_n = 100 # Neighbour/interaction radius
bounds = 400 # Simulation bounds!

k = 5 # Force constant 

divs = r_n # Edge length of each cell

t_h = [0] # Time history
k_h = [0] # Kinetic energy history

$\text{Atom is defined by the following data scheme:}$

$$
\begin{bmatrix}
x \\
y \\
z \\
v_x \\
v_y \\
v_z \\
x_{\text{cell}} \\
y_{\text{cell}} \\
z_{\text{cell}} \\
\end{bmatrix}
$$

$\text{where}$

$$
x,y,z \rightarrow \text{Components of position in cartesian coordinates} \\
v_i \rightarrow \text{Component of velocity in i-th direction} \\
x_{\text{cell}}, y_{\text{cell}}, z_{\text{cell}} \rightarrow \text{The indices of the cell containing the atom}
$$

Also, 

$m$ is the mass of the atom


In [22]:
# This generates a 7xn array, since we have n atoms with 7 data points each
# TODO: We REALLY gotta add proper initial value setting here
atoms = np.zeros((n, 9))

s=100

# Source - https://stackoverflow.com/a/33977530
# Posted by ali_m, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-15, License - CC BY-SA 3.0

import numpy as np

def sample_spherical(npoints, ndim=3):
    vec = np.random.randn(ndim, npoints)
    vec /= np.linalg.norm(vec, axis=0)
    return vec

from math import floor
atoms[:, :3] = (np.vectorize(floor))(np.transpose(sample_spherical(n))*100)
atoms[:, 3:6] = np.transpose(sample_spherical(n))*10
atoms[:, 6:] = atoms[:, :3] // divs

Now we define some helper functions here that will see heavy use:

- `cellOf(a)`: Finds the current cell of the specified atom `a`.


In [ ]:
def updateCell_(a):

    # We use the dims array we initialized in the beginning
    global divs

    # Creating a list of basically [(0, dims[0]), (1, dims[1]), (2, dims[2])
    a[6:] = a[:3] // divs
    
def getNeigbours(a):
    #Using the stuff we initialized before
    global r_n, atoms

    # We get all the cells that are at most 1 cell away from the current cell in every direction.
    # This defines a 5x5x5 cuboid containing the cell of a at its center, and cuts
    # down on any extra unnecessary computation.
    search_space = np.array(
        [k for k in atoms if
        (np.abs((k-a)[6:]) < np.array([2, 2, 2])).all()])

    # And then we find the neighbours of the atom in the current cell.
    neighbours = np.array(
        [k for k in search_space if ( (np.linalg.norm(a[:3] - k[:3]) < r_n) and (a != k).any())])
    return neighbours


def force_func(r):
    return -1*k*(r - r_n)

#                     <- Force as func of r ->   <- Unit dir. vector ->
force = lambda x, y : (force_func(la.norm(x-y)) * (x-y) / (la.norm(x-y)))

def calculateForces(a):
    neighbours = getNeigbours(a)
    if len(neighbours) > 1 and (neighbours[:, :3]!= np.array([a])[:, :3]).all():
        f = np.sum(force(neighbours[:, :3], np.array([a])[:, :3]), axis=0)
    else:
        f = np.zeros(3)
    return f



def step():
    
    global t_h, k_h

    for j in range(3):
        nex = abs(atoms[:, j] + (atoms[:, j+3] * dt)) > 400
        atoms[:, j+3] *= (np.vectorize(lambda x: -1 if x else 1))(nex)

    

    atoms[:, 0:3] += atoms[:, 3:6] * dt
    f = np.array([calculateForces(x) for x in atoms])

    # TODO: Implement usage of particle mass here
    atoms[:, 3:6] += f * dt * (1/4)


    




In [24]:
%matplotlib qt5

fig = plt.figure()


ui_axes = [ (0.1, 0.5, 0.4, 0.4),  (0.1, 0.2, 0.4, 0.05), (0.1, 0.1, 0.4, 0.05)]

ax = fig.add_subplot(ui_axes[0],projection='3d')
ax_ke = fig.add_axes((0.6, 0.6, 0.3, 0.3))
ax_k_s = fig.add_axes(ui_axes[1])
ax_r_n_s = fig.add_axes(ui_axes[2])
ke_line, = ax_ke.plot([],[], lw = 1.5)


k_s = Slider(ax_k_s, "Force constant", 1, 20, valinit=5)
r_n_s = Slider(ax_r_n_s, "Interaction radius", 100, bounds/2, valinit=100)



def k_changed(x):
    global k
    k = x

def r_n_changed(x):
    global r_n
    r_n = x

k_s.on_changed(k_changed)
r_n_s.on_changed(r_n_changed)


from matplotlib.animation import FuncAnimation
ax.autoscale()

def animate(i):
    
    ax.clear()
    ax.set_ylim(-1*bounds, bounds)
    ax.set_xlim(-1*bounds, bounds)
    ax.set_zlim(-1*bounds, bounds)
    ax_ke.set_xbound(0, t_h[-1])
    ax_ke.autoscale(enable=True)
    
    
    t_h.append(t_h[-1] + dt)
    k_h.append(np.sum(atoms[:, 3:6] ** 2))

    if(ax_ke.get_ybound()[1] < k_h[-1]*1.5):
        ax_ke.set_ybound(0, k_h[-1]*1.5)


    print(k_h)
    ke_line.set_data(t_h, k_h)

    
    ax.scatter(atoms[:, 0], atoms[:, 1], atoms[:, 2], color = "red", marker = "o" )
    # k = k_s.value
    # r_n = r_n_s.value
    step()
    

anim = FuncAnimation(fig, animate,frames=600, interval=50)

plt.show()

qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
